# 📚 3.4 — Choosing Activations — Revision Notes

This section is about **choosing the correct activation based on the role of the layer and the ML task**.

---

# 1. Core Decision Rule 🔥

Always think:

$$
\boxed{
\text{Task}
\rightarrow
\text{Output Structure}
\rightarrow
\text{Activation}
\rightarrow
\text{Loss}
}
$$

For hidden layers, the choice is usually based on the architecture.

---

# 2. Hidden Layers

Hidden layers need **non-linearity** so the network can learn complex relationships.

## Common choices

### ReLU

$$
ReLU(z)=\max(0,z)
$$

Common starting choice for:

* Basic MLPs
* Many CNNs
* Traditional deep neural networks

### GELU

$$
GELU(x)=x\Phi(x)
$$

Commonly encountered in:

* Transformers
* Modern neural architectures

### Leaky ReLU

Useful as a ReLU alternative when retaining a small negative-side gradient is desirable.

---

## Hidden-layer rule

$$
\boxed{\text{Hidden layers → ReLU / GELU / suitable variant}}
$$

There is **no universal best activation**.

Architecture and training behavior matter.

---

# 3. Binary Classification

Examples:

* Spam / not spam
* Pass / fail
* Disease / no disease
* Cat / not cat

Usually:

$$
\boxed{1\text{ output logit}}
$$

Conceptually:

$$
z\rightarrow sigmoid(z)\rightarrow probability
$$

Sigmoid:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Example:

$$
z=2.2
$$

gives approximately:

$$
\sigma(2.2)\approx0.90
$$

So the model predicts approximately 90% probability for the positive class.

### PyTorch

Use:

```python
nn.BCEWithLogitsLoss()
```

The model should normally output **raw logits**.

If probabilities are needed during inference:

```python
probability = torch.sigmoid(logits)
```

### Rule

$$
\boxed{\text{Binary → 1 output logit → sigmoid conceptually}}
$$

---

# 4. Multiclass Classification

Examples:

```text
Cat
Dog
Horse
```

Only **one** class is correct.

For $K$ classes:

$$
\boxed{K\text{ output logits}}
$$

Conceptually:

$$
P_i=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

This is **softmax**.

Example:

```text
Cat    → 0.72
Dog    → 0.19
Horse  → 0.09
```

The probabilities sum to:

$$
1
$$

### PyTorch

Use:

```python
nn.CrossEntropyLoss()
```

The model should normally output **raw logits**.

### Rule

$$
\boxed{\text{Multiclass → K output logits → softmax conceptually}}
$$

---

# 5. Regression

Regression predicts a continuous value.

Examples:

* House price
* Temperature
* Motor temperature
* Torque
* Energy consumption
* Battery-related continuous values

Usually the output is:

$$
\boxed{\text{Linear / no activation}}
$$

So:

$$
z=Wx+b
$$

and:

$$
\hat y=z
$$

Example:

```text
Input
 ↓
Neural Network
 ↓
Linear output
 ↓
72.4
```

### Why not sigmoid?

Sigmoid restricts output to:

$$
(0,1)
$$

which is usually inappropriate for unrestricted continuous targets.

### Common losses

* MSE
* MAE
* Huber

### Rule

$$
\boxed{\text{Regression → Linear/no output activation}}
$$

---

# 6. Multi-Label Classification

This is important because it is easy to confuse with multiclass.

Suppose an image contains:

```text
☑ Dog
☑ Car
☑ Person
```

Multiple classes can be true simultaneously.

Use:

$$
\boxed{\text{Independent sigmoid outputs}}
$$

For $K$ labels:

$$
K\text{ output logits}
$$

Each output is independently converted using sigmoid.

### PyTorch

```python
nn.BCEWithLogitsLoss()
```

### Rule

$$
\boxed{\text{Multi-label → K logits → independent sigmoid conceptually}}
$$

---

# 🔥 7. Master Comparison

| Task                           |    Output | Conceptual activation | Common loss         |
| ------------------------------ | --------: | --------------------- | ------------------- |
| **Regression**                 | Usually 1 | Linear / none         | MSE / MAE / Huber   |
| **Binary classification**      |         1 | Sigmoid               | `BCEWithLogitsLoss` |
| **Multiclass classification**  |       $K$ | Softmax               | `CrossEntropyLoss`  |
| **Multi-label classification** |       $K$ | Independent sigmoid   | `BCEWithLogitsLoss` |

---

# 🔥 8. Hidden vs Output Activation

This distinction is extremely important.

### Hidden layers

Their job is to learn useful representations:

```text
Linear
 ↓
ReLU/GELU
 ↓
Linear
 ↓
ReLU/GELU
```

### Output layer

Its activation depends on the **task**:

```text
Regression      → Linear
Binary          → Sigmoid conceptually
Multiclass      → Softmax conceptually
Multi-label     → Independent Sigmoids
```

---

# ⚠️ 9. Important PyTorch Detail

Don't confuse the **conceptual activation** with what you explicitly put inside the model.

### Binary

Conceptually:

$$
logit\rightarrow sigmoid\rightarrow probability
$$

But commonly:

```python
model → raw logits
       ↓
BCEWithLogitsLoss
```

### Multiclass

Conceptually:

$$
logits\rightarrow softmax\rightarrow probabilities
$$

But commonly:

```python
model → raw logits
       ↓
CrossEntropyLoss
```

### Why?

These PyTorch losses combine the necessary operations in a numerically stable way.

So:

> 🚨 **Don't manually apply sigmoid before `BCEWithLogitsLoss`.**

> 🚨 **Don't manually apply softmax before `CrossEntropyLoss`.**

---

# 🧠 10. Quick Decision Tree

```text
                 WHAT IS THE TASK?
                        │
        ┌───────────────┼────────────────┐
        ↓               ↓                ↓
   Regression        Binary          Multiclass
        │               │                │
        ↓               ↓                ↓
 Linear/None       1 output          K outputs
                        │                │
                        ↓                ↓
                  Sigmoid concept    Softmax concept
```

And:

```text
Multiple labels can be true?
            │
           YES
            ↓
     K independent outputs
            ↓
     Independent sigmoid
```

---

# 🎯 11. Interview Cheat Sheet

### Q: What activation do you use in hidden layers?

> ReLU is a common starting choice for basic neural networks; GELU is common in Transformer-style architectures. The architecture determines the appropriate choice.

### Q: Binary classification?

> One output logit with sigmoid conceptually; commonly train using `BCEWithLogitsLoss`.

### Q: Multiclass classification?

> $K$ output logits with softmax conceptually; commonly train using `CrossEntropyLoss`.

### Q: Regression?

> Linear/no output activation.

### Q: Multi-label classification?

> $K$ independent logits with independent sigmoid outputs conceptually; commonly use `BCEWithLogitsLoss`.

---

# 🏆 Final Memory Trick

> **Hidden → ReLU/GELU**
> **Binary → Sigmoid**
> **Multiclass → Softmax**
> **Multi-label → Independent Sigmoids**
> **Regression → Linear**

And the most important design chain:

$$
\boxed{
\text{Task}
\rightarrow
\text{Output}
\rightarrow
\text{Activation}
\rightarrow
\text{Loss}
}
$$
